In [32]:
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
import keras
from keras import regularizers
import tensorflow as tf
from keras import layers


regr = HistGradientBoostingRegressor(max_depth=24,learning_rate=0.05,early_stopping=True,max_iter=500)
model = keras.Sequential([
    layers.Dense(256, activation="relu",kernel_regularizer=regularizers.L1(0.01)),
    layers.Dense(128, activation="relu",kernel_regularizer=regularizers.L1(0.01)),
    layers.Dense(64, activation="relu",kernel_regularizer=regularizers.L1(0.01)),
    layers.Dense(1)
    
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss="mean_absolute_error")
train_dataset = pd.read_csv("Datasets/train.csv")
test_dataset = pd.read_csv("Datasets/test.csv")

train_set = train_dataset.drop(["Id"],axis=1)
test_set = test_dataset.drop(["Id"],axis=1)
test_Y = test_dataset["Id"]
train_set["LotFrontage"] = train_set["LotFrontage"].fillna(0)
train_set["MasVnrArea"] = train_set["MasVnrArea"].fillna(train_set["MasVnrArea"].median())
train_set["GarageYrBlt"] = train_set["GarageYrBlt"].fillna(train_set["GarageYrBlt"].median())
test_set["LotFrontage"] = test_set["LotFrontage"].fillna(0)
test_set["MasVnrArea"] = test_set["MasVnrArea"].fillna(test_set["MasVnrArea"].median())
test_set["GarageYrBlt"] = test_set["GarageYrBlt"].fillna(test_set["GarageYrBlt"].median())
onehot = OneHotEncoder(sparse=False,handle_unknown="ignore")
new_columns_train = pd.DataFrame(onehot.fit_transform(train_set[["GarageQual","GarageCond","KitchenQual","CentralAir","PavedDrive","MSZoning","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","RoofStyle",
                                  "RoofMatl","Exterior1st","Exterior2nd","MasVnrType","ExterQual","ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2","Heating",
                                  "HeatingQC","Electrical","Functional","FireplaceQu","GarageType","GarageFinish","PoolQC","Fence","MiscFeature","SaleType","SaleCondition"]]),columns = onehot.get_feature_names_out(),index=train_dataset.index)
train_after_onehot = train_set.drop(columns=["GarageQual","GarageCond","KitchenQual","CentralAir","PavedDrive","MSZoning","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","RoofStyle",
                                  "RoofMatl","Exterior1st","Exterior2nd","MasVnrType","ExterQual","ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2","Heating",
                                  "HeatingQC","Electrical","Functional","FireplaceQu","GarageType","GarageFinish","PoolQC","Fence","MiscFeature","SaleType","SaleCondition"],axis=1).join(new_columns_train)
new_columns_test = pd.DataFrame(onehot.transform(test_set[["GarageQual","GarageCond","KitchenQual","CentralAir","PavedDrive","MSZoning","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","RoofStyle",
                                  "RoofMatl","Exterior1st","Exterior2nd","MasVnrType","ExterQual","ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2","Heating",
                                  "HeatingQC","Electrical","Functional","FireplaceQu","GarageType","GarageFinish","PoolQC","Fence","MiscFeature","SaleType","SaleCondition"]]),columns = onehot.get_feature_names_out(),index=test_dataset.index)
test_after_onehot = test_set.drop(columns=["GarageQual","GarageCond","KitchenQual","CentralAir","PavedDrive","MSZoning","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","RoofStyle",
                                  "RoofMatl","Exterior1st","Exterior2nd","MasVnrType","ExterQual","ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2","Heating",
                                  "HeatingQC","Electrical","Functional","FireplaceQu","GarageType","GarageFinish","PoolQC","Fence","MiscFeature","SaleType","SaleCondition"],axis=1).join(new_columns_test)
X = train_after_onehot.drop(["SalePrice"],axis=1)
Y = train_after_onehot["SalePrice"]
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size = 0.25,random_state=42)
early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True)
regr.fit(X_train,Y_train)
model.fit(X_train,Y_train,epochs=100,batch_size=120,validation_data=(X_val,Y_val),callbacks=early_stopper)
print(regr.score(X_val, Y_val))
print(model.evaluate(X_val,Y_val))
result_test = regr.predict(test_after_onehot)
result_test_NN = model.predict(test_after_onehot)
result_test_ensemble = (result_test * 0.5) + (result_test_NN[:,0] * 0.5)

result = pd.DataFrame({"Id":test_Y,"SalePrice":result_test},columns=["Id","SalePrice"])
result_NN = pd.DataFrame({"Id":test_Y,"SalePrice":result_test_NN[:,0]},columns=["Id","SalePrice"])
result_Ensemble = pd.DataFrame({"Id":test_Y,"SalePrice":result_test_ensemble},columns=["Id","SalePrice"])
result.to_csv("submission.csv",index=False)
result_NN.to_csv("submissionNN.csv",index=False)
result_Ensemble.to_csv("submissionEnseble.csv",index=False)

Epoch 1/100
10/10 [==============================] - 1s 16ms/step - loss: 163106.0781 - val_loss: 84207.8672
Epoch 2/100
10/10 [==============================] - 0s 5ms/step - loss: 66950.4922 - val_loss: 54978.6836
Epoch 3/100
10/10 [==============================] - 0s 6ms/step - loss: 54227.5820 - val_loss: 46776.9648
Epoch 4/100
10/10 [==============================] - 0s 6ms/step - loss: 46634.2891 - val_loss: 43038.9102
Epoch 5/100
10/10 [==============================] - 0s 5ms/step - loss: 43135.4609 - val_loss: 38750.0234
Epoch 6/100
10/10 [==============================] - 0s 6ms/step - loss: 38572.7734 - val_loss: 34809.1602
Epoch 7/100
10/10 [==============================] - 0s 6ms/step - loss: 36693.3750 - val_loss: 33185.7266
Epoch 8/100
10/10 [==============================] - 0s 5ms/step - loss: 35304.3906 - val_loss: 35350.4727
Epoch 9/100
10/10 [==============================] - 0s 6ms/step - loss: 33937.8438 - val_loss: 36064.4102
Epoch 10/100
10/10 [===============